# Reconstructed analysis walkthrough

This notebook is **not** a recovered Spring 2026 original. It is a September 2026
walkthrough of the published CompSci 390B methodology.

- Published numbers in markdown tables are from the May 2026 report.
- Code cells call `src/macro_stress`, which reconstructs the methodology with
  library defaults. Those outputs are labeled reconstructed.
- If `data/Global_Market_Stress_and_Liquidity_Regimes.csv` is missing, later
  cells stop after a data check instead of inventing results.

See `docs/RECOVERY_NOTES.md` and `docs/RESULTS.md`.


## 1. Setup and data

Acquire the CSV with `python scripts/download_data.py` (needs `~/.kaggle/kaggle.json`)
or a manual Kaggle download. The loader always restricts to 2014-10-17 through 2026-02-25.


In [ ]:
from pathlib import Path
import json
import pandas as pd

from macro_stress.data import data_check, find_dataset, load_raw, validate_dataset
from macro_stress.features import FEATURE_COUNT, build_supervised_frame, drop_unusable, feature_names, log_returns
from macro_stress.regimes import HISTORICAL_THRESHOLDS, add_stress_flags, compute_thresholds

info = data_check()
print(json.dumps(info, indent=2))
DATA_OK = bool(info.get("found")) and not info.get("issues")
print("DATA_OK", DATA_OK)


## 2. Log returns and stress flags

Published 66th-percentile thresholds (checksums): HYS 4.400, FSI −0.172, VIX 19.060.
Shares of days: hys 34.2%, fsi 34.1%, vix 34.0%, any 56.3%, all 14.3%.


In [ ]:
if not DATA_OK:
    print("Skipping remaining cells — place the Kaggle CSV in data/ and re-run.")
else:
    raw = load_raw(find_dataset())
    print("restricted rows", len(raw), raw["Date"].min().date(), "→", raw["Date"].max().date())
    print("Gold log-return check (first valid):")
    r = log_returns(raw["Gold"])
    print(float(r.iloc[1]), float(__import__("numpy").log(raw["Gold"].iloc[1]) - __import__("numpy").log(raw["Gold"].iloc[0])))
    flagged = add_stress_flags(raw)
    computed = compute_thresholds(raw)
    print("computed thresholds", {k: round(v, 3) for k, v in computed.items()})
    print("historical checksums", HISTORICAL_THRESHOLDS)
    for flag in ("stress_hys", "stress_fsi", "stress_vix", "stress_any", "stress_all"):
        print(flag, f"{flagged[flag].mean():.3%}")


## 3. Feature matrix (reconstructed 55 columns)

The report names feature *families*, not the exact list. `feature_names()` is a
documented reconstruction that includes every named feature and yields 55 columns.


In [ ]:
print("FEATURE_COUNT", FEATURE_COUNT)
print("named in the report:")
for name in ("spy_ret_lag1", "spy_ret_lag3", "gold_ret_lag1", "gold_ret_lag3",
             "High_Yield_Spread_lag5", "Financial_Stress_Index_lag5",
             "Volatility_Index", "GLD_RSI_14", "stress_any_t"):
    print(" ", name, name in feature_names())
if DATA_OK:
    frame = build_supervised_frame(raw)
    usable = drop_unusable(frame, extra=("y_gold_gt_spy",))
    print("first usable", usable["Date"].min().date(), "n", len(usable))
    print("target is next-day Gold > SPY; last raw day has no target")


## 4. Published safe-haven results (report checksums)

Do not replace these with reconstructed means.

| Regime | Gold–SPY widening | CI lower |
|---|---|---|
| stress_vix | +0.00247 | +0.00157 |
| stress_any | +0.00096 | +0.00029 |
| stress_all | +0.00191 | +0.00040 |
| stress_fsi | −0.00012 | no Gold advantage |

Under `stress_vix`: Gold +0.00068, SPY −0.00092, BTC −0.00008.


In [ ]:
if DATA_OK:
    from macro_stress.analysis import mean_return_table, vol_by_regime, spy_beta
    panel = drop_unusable(frame, extra=("gold_ret", "spy_ret"))
    print("RECONSTRUCTED regime table (not a replacement for the report):")
    for flag in ("stress_vix", "stress_any", "stress_all", "stress_fsi", "stress_hys"):
        stats = mean_return_table(panel, flag)
        print(flag, {k: round(v, 5) if isinstance(v, float) else v for k, v in stats.items()})


## 5. Models E / F / G — published checksums

| Model E | ROC–AUC |
|---|---|
| Majority | 0.5000 |
| VIX rule | 0.5040 |
| Logistic | 0.6091 |
| Random Forest | 0.6622 |
| LightGBM | 0.6687 |
| CatBoost | 0.6722 |
| Stacking | 0.6800 |
| **XGBoost** | **0.6835** |

Model F: Ridge beats XGBoost on SPY 30d vol h=5 (0.0210 vs 0.0246). XGBoost VIX h=5 R² = −0.4337.

Model G: logistic PR–AUC **0.7095** vs majority 0.2433.

Walk-forward mean ROC–AUC **0.6571** (σ 0.0289).

A reconstructed fit uses library defaults and will usually miss those numbers.
That is expected. Do not tune to chase them. Run `make reproduce` for a full
reconstructed dump to `results/latest/`.


In [ ]:
print("To run the reconstructed pipelines (slow; needs the CSV):")
print("  python -m macro_stress.cli reproduce")
print("Published checksums live in results/historical/anchors.json")
print(Path("results/historical/anchors.json").read_text()[:600])
